# Tanga 3D Visualization — Interactive Notebook

This notebook demonstrates the `pytanga.viz` submodule using **N3 conformal geometric algebra**.
We'll create spheres in IPNS form, compute their intersections via the outer product $\wedge$,
and view everything in an interactive 3D viewer embedded directly in the notebook.

## Geometry background (N3 / CGA)

In N3 conformal GA, **IPNS spheres** are grade-1 multivectors (vectors):

$$S = o + c + \tfrac{1}{2}(c^2 - r^2)\,e_\infty$$

where $c$ is the center point and $r$ is the radius.

The **intersection of two spheres** $S_1 \wedge S_2$ is a **circle** (or imaginary circle),
and $S_1 \wedge S_2 \wedge S_3$ is a **point pair**.

In [ ]:
# ── Imports ──────────────────────────────────────────
from pytanga.basis import BasisN3
from pytanga.geometry import Geometry, Point, Sphere
from pytanga.viz import Visualizer

print("✓ Imports ready")

## 1. Start the viewer

The server runs in a background daemon thread so the kernel stays responsive.
In a notebook we use `start_server()` / `flush()` / `stop_server()` — **never** `run()` which blocks.

In [ ]:
viz = Visualizer()
viz.start_server()
print(f"Server running at {viz.url}")

## 2. Create three IPNS spheres

Using `N3` basis with `opns=False` (IPNS interpretation).

In [ ]:
N3 = BasisN3(opns=False)
geo = Geometry(N3)

# Three overlapping unit spheres at different positions
S1 = geo(Sphere(Point(0.0, 0.0, 0.0), 1.0))
S2 = geo(Sphere(Point(1.0, 0.0, 0.0), 1.0))
S3 = geo(Sphere(Point(0.5, 1.0, 0.0), 1.0))

print(f"S₁ MV: {S1}")
print(f"S₂ MV: {S2}")
print(f"S₃ MV: {S3}")

## 3. Analyze what each MV represents

The `which_entity()` method introspects the blade structure and returns the
corresponding geometric entity object.

In [ ]:
for name, mv in [("S₁", S1), ("S₂", S2), ("S₃", S3)]:
    # You could also use entity = geo(mv) to get the entity from the multivector
    entity = geo.which_entity(mv)
    print(f"  {name} → {entity}")

## 4. Compute intersections via the outer product

In IPNS, the outer product $\wedge$ corresponds to the **intersection** (meet) of objects:

- $S_1 \wedge S_2$ = **circle** (intersection of two spheres)
- $S_1 \wedge S_2 \wedge S_3$ = **point pair** (intersection of three spheres)

In [ ]:
# Two-sphere intersection → circle
C1 = S1 ^ S2  # circle
c1_entity = geo(C1)
print(f"  S₁ ∧ S₂ → {c1_entity}")

# Three-sphere intersection → point pair
PP1 = C1 ^ S3  # point pair
pp1_entity = geo(PP1)
print(f"  S₁ ∧ S₂ ∧ S₃ → {pp1_entity}")

## 5. Add everything to the scene

In [ ]:
# Spheres — semi-transparent

s1_viz = viz.new(S1, color="#ff4444", label="$S_1$")
s2_viz = viz.new(S2, color="#3620de", opacity="0.4", label="$S_2$")
s3_viz = viz.new(S3, color="#dd44ff", label="$S_3$")

# Intersection circle
c1_viz = viz.new(C1, color="#D1BF1D", label="$S_1\\land S_2$")

# Intersection point pair
pp1_viz = viz.new(PP1, color="#1AB03D", label="$S_1\\land S_2\\land S_3$")

# Push to the browser
viz.flush()
print("✓ Scene updated")

## 6. View the scene inline

When `viz` is the last expression in a cell, Jupyter calls `_repr_html_()`
which embeds the viewer in an iframe.

💡 **Tip:** You can orbit (left-drag), pan (right-drag), and zoom (scroll) in the viewer.

In [ ]:
# viz.display_snapshot()

## 7. Keyframe animation: move a sphere

`animate_to()` sends a tween command to the browser. The animation runs
smoothly on the GPU — no Python loop needed.

In [ ]:
import time

# Add a standalone point to animate
pt_viz = viz.new(
    Point(-2, -1, 0),
    color="#44ff44",
    label="P",
)
viz.flush()
time.sleep(0.5)

# Slide the point from (-2, -1, 0) → (3, 2, 2) over 2 seconds
viz.animate_to(pt_viz.id, position=(3, 2, 2), duration=2.0, easing="ease-in-out")
print("✓ Animation dispatched — watch the point move in the viewer above")

## 8. Timeline: sequenced animations

A `Timeline` schedules multiple tweens with delays and optional parallel execution.

In [ ]:
(
    viz.timeline()
    .animate_to(pt_viz.id, position=(-2, -1, 0), duration=1.0)
    .wait(0.3)
    .animate_to(s2_viz.id, opacity=0.15, duration=1.5, easing="ease-out")
    .wait(0.3)
    .animate_to(s2_viz.id, opacity=0.4, duration=1.0)
    .play()
)
print("✓ Timeline dispatched")

## 9. Frame streaming: animate an intersection

For continuous per-frame updates (e.g., a moving sphere with its intersection
circle recalculated every frame), use `update_entity()` + `flush()`.

In [ ]:
import math

print("Animating S₂ along the x-axis for 10 seconds …")
t_start = time.monotonic()
while (time.monotonic() - t_start) < 10.0:
    elapsed = time.monotonic() - t_start
    x = 3.0 * math.sin(elapsed * 1.2)  # oscillate between -3 and +3

    # Update S₂ position
    s2_new = Sphere(Point(x, 0.5, 0.0), 1.0)
    s2_new_mv = geo.create(s2_new)
    s2_viz.entity = s2_new

    # Recompute intersection circle
    C_new = S1 ^ s2_new_mv
    c1_viz.entity = C_new

    viz.flush()
    time.sleep(1.0 / 60.0)

print("✓ Animation complete")

In [ ]:
viz

## 10. Cleanup

Always stop the server when you're done to free the port.

In [ ]:
viz.stop_server()
print("✓ Server stopped")

---

## Bonus: Available entity & operator types

| Entity | Description |
|--------|-------------|
| `Point(x, y, z)` | Euclidean point |
| `Direction(x, y, z)` | Direction vector (visualized as arrow) |
| `HPoint(point, weight)` | Homogeneous point |
| `PointPair(a, b)` | Point pair (line segment) |
| `Line(origin, direction)` | Infinite line |
| `Plane(point, normal)` | Infinite plane |
| `Circle(center, normal, radius)` | Circle |
| `Sphere(center, radius)` | Sphere |
| `Space(scale)` | Full 3D space (pseudoscalar outline) |

| Operator | Description |
|----------|-------------|
| `Rotor(angle, axis)` | Rotation bivector |
| `Translator(dx, dy, dz)` | Translation vector |
| `Motor(rotor, translator)` | Rigid motion = rotation + translation |
| `ReflectionPlane(normal)` | Reflection in a plane |
| `ReflectionLine(direction)` | Reflection on a line |
| `ReflectionOrigin()` | Point reflection |
| `Inversion(center, radius)` | Sphere inversion |
| `Dilator(factor)` | Uniform scaling |
| `GeneralRotor(rotor, translator)` | Rotor translated to arbitrary point |
| `GeneralDilator(factor, translator)` | Dilator translated to arbitrary point |

### Rendering properties (`ObjVizProps`)

- `color="#rrggbb"` — hex color or RGB/RGBA float tuple
- `opacity=0.0..1.0` — transparency
- `style=PointStyle(size=...)` — entity-specific size/thickness

### Animation API

- `viz.animate_to(id, *, position, opacity, duration, easing)` — single tween
- `viz.timeline().animate_to(...).wait(...).play()` — sequenced tweens
- `viz.update_entity(id, new_obj)` + `viz.flush()` — frame streaming

### Easing functions: `"linear"`, `"ease-in"`, `"ease-out"`, `"ease-in-out"`